In [1]:
# Configuración de entorno
ENTORNO = "local" # Opciones: "local" o "colab"
ALMACENAMIENTO = "local" # Opciones: "local" o "drive" (solo aplica si ENTORNO = "colab")

# Rutas base según la configuración
RUTA_DRIVE = '/content/drive/MyDrive/gnn-material-science'
RUTA_LOCAL = '..' if os.path.basename(os.getcwd()) == 'MACE' else '.' # Ajuste automático de directorio raíz

import sys

if ENTORNO == "colab":
    print("Ejecutando en entorno Google Colab...")
    dispositivo = "cuda"
    if ALMACENAMIENTO == "drive":
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            os.chdir(RUTA_DRIVE)
            print(f"Montado Drive y trabajando en {RUTA_DRIVE}")
        except ImportError:
            print("No se pudo montar drive. ¿Seguro que estás en Colab?")
    else:
        print("Usando almacenamiento local de la sesión de Colab.")
    
    # Instalamos dependencias si estamos en Colab
    print("Comprobando e instalando dependencias (puede tardar un poco)...")
    get_ipython().system('pip install -q -r requirements.txt')
    get_ipython().system('pip install -q cuequivariance cuequivariance-torch cuequivariance-ops-torch-cu12')
elif ENTORNO == "local":
    os.chdir(RUTA_LOCAL)
    dispositivo = "cpu" # Forzamos CPU para local según especificaciones
    print(f"Ejecutando en entorno local usando {dispositivo.upper()}. Directorio actual: {os.getcwd()}")
    if 'google.colab' in sys.modules:
        print("ADVERTENCIA: Parece que estás en Colab pero has seleccionado entorno 'local'.")
    
    print("Comprobando e instalando dependencias locales (puede tardar un poco)...")
    get_ipython().system('pip install -q -r requirements.txt')
else:
    print("Entorno no reconocido. Usa 'colab' o 'local'.")
    dispositivo = "cpu"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/gnn-material-science


In [2]:
from mp_api.client import MPRester
from pathlib import Path
from pymatgen.io.vasp import Poscar
import json
import os
import re

from dotenv import load_dotenv
load_dotenv()
MP_ID = os.getenv('MP_API_KEY')

output_name = "input/candidates"
output_dir = Path(output_name)
output_dir.mkdir(parents=True, exist_ok=True)

In [3]:
def generate_POSCAR(path_to_folder, doc):
    """Create POSCAR and metadata.json in formula/symmetry folder."""

    # Root folder
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)

    # Formula folder
    path_to_folder = f"{path_to_folder}/{doc.formula_pretty}"
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)

    # Symmetry folder
    symmetry_folder = re.sub('/', '-', doc.symmetry.symbol)
    path_to_folder = f"{path_to_folder}/{symmetry_folder}"
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)

    # ---------- POSCAR ----------
    poscar_path = f"{path_to_folder}/POSCAR"
    poscar = Poscar(doc.structure)
    poscar.comment = f"{doc.formula_pretty} ({doc.material_id})"
    poscar.write_file(poscar_path)

    # ---------- metadata ----------
    metadata = {
        "material_id": str(doc.material_id),
        "formula": doc.formula_pretty,
        "symmetry": doc.symmetry.symbol,
        "elements": [str(el) for el in doc.elements],
        "nelements": doc.nelements,
        "volume": doc.volume,
        "density": doc.density,
        "energy_above_hull": doc.energy_above_hull,
        "formation_energy_per_atom": doc.formation_energy_per_atom,
        "band_gap": doc.band_gap,
        "is_stable": doc.is_stable
    }

    json_path = f"{path_to_folder}/metadata.json"
    with open(json_path, "w") as f:
        json.dump(metadata, f, indent=2)

In [4]:
elements = [
    'H','He','Li','Be','B','C','N','O','F','Ne','Na','Mg','Al','Si','P','S','Cl',
    'Ar','K','Ca','Ti','V','Cr','Mn','Fe','Co','Ni','Cu','Zn','Ga','Ge','As','Se','Br',
    'Kr','Sr','Y','Zr','Nb','Mo','Rh','Ag','Cd','In','Sn','Sb','Te','I','Ba','La','Ce',
    'Pr','Nd','Sm','Eu','Gd','Tb','Dy','Ho','Er','Yb','Lu','Hf','Ta','W','Hg','Pb','Bi',
    'At','Rn','Fr','Ra','Pa','U','Es','Fm','Md','No','Lr','Rf','Db','Sg','Bh','Hs','Mt',
    'Ds','Rg','Cn','Nh','Fl','Mc','Lv','Ts','Og','Cs','Pd','Au','Xe'
]

"""
exclude_elements = [
    'Po','Ac','Tc','Cf','Bk','Cm','Pu','Am','Np','Pm',
    'Ir','Pt','Rb','Os','Ru','Tl','Sc','Re','Th','Tm'
]
"""

exclude_elements = [
    "Tc", "Pm",   # muy raros
    "Po", "At", "Rn",  # problemáticos/químicamente raros
    "Fr", "Ra",
    "Ac", "Th", "Pa", "U",
    "Np", "Pu", "Am", "Cm", "Bk", "Cf"
]

"""
fields = [
    "material_id",
    "formula_pretty",
    "structure",
    "symmetry",
    "energy_per_atom",
    "band_gap"
]
"""

"""
fields = [
    "material_id",
    "formula_pretty",
    "structure",
    "symmetry",
    "formation_energy_per_atom", # Cambiado por energy_per_atom
    "e_above_hull",              # ¡Nuevo e importante!
    "is_stable",                 # ¡Nuevo!
    "theoretical",               # ¡Nuevo!
    "band_gap",
    "density",                   # ¡Nuevo!
    "volume"                     # ¡Nuevo!
]
"""

fields = [
    "material_id",
    "formula_pretty",
    "structure",
    "elements",
    "nelements",
    "volume",
    "density",
    "energy_above_hull",
    "formation_energy_per_atom",
    "band_gap",
    "is_stable",
    "symmetry"
]

print("Querying Materials Project...")

with MPRester(MP_ID) as mpr:
    docs = mpr.materials.summary.search(
        elements=['Li'], # De momento solo buscamos materiales con litio, pero se pueden añadir más elementos
        exclude_elements=exclude_elements,
        fields=fields,
        energy_above_hull=(0, 0.05), # Solo materiales que estén a menos de 50 meV/atom de la convex hull (es decir, potencialmente estables)
        band_gap=(0.5, None),  # Solo materiales con band gap > 0.5 eV (para asegurar que sean aislantes o semiconductores)
        all_fields=False
    )

print(f"Retrieved {len(docs)} materials")

Querying Materials Project...


Retrieving SummaryDoc documents:   0%|          | 0/6063 [00:00<?, ?it/s]

Retrieved 6063 materials


In [5]:
for doc in docs:
    try:
        if doc.structure is None:
            continue

        generate_POSCAR(output_name, doc)

    except Exception as e:
        print("Skip", doc.material_id, e)

KeyboardInterrupt: 